In [ ]:
import json
import os
from sklearn.model_selection import train_test_split
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
from tqdm import tqdm


In [ ]:
input_file_path = r"C:\Users\28995\Desktop\Heavenly Lake -2\train.json"
output_dir = r"C:\Users\28995\Desktop\Heavenly Lake -2\split_output" # 固定文件夹名，不再带时间戳

# 确保输出文件夹存在
os.makedirs(output_dir, exist_ok=True)

# 2. 加载原始数据
if os.path.exists(input_file_path):
    with open(input_file_path, 'r', encoding='utf-8') as f:
        all_data = json.load(f)
    print(f"✅ 数据加载成功！共包含 {len(all_data)} 条数据。")
else:
    print(f"❌ 错误：未找到 {input_file_path}，请检查路径是否正确。")
    exit()

# 3. 数据切分 (8:2)
train_data, val_data = train_test_split(all_data, test_size=0.2, random_state=42)
print(f"📋 示例库 (train_data) 大小: {len(train_data)} 条")
print(f"🧪 验证集 (val_data) 大小: {len(val_data)} 条")

# 4. 保存为指定名称的 JSON 文件
try:
    # 保存为 train_data.json
    train_output_path = os.path.join(output_dir, "train_data.json")
    with open(train_output_path, 'w', encoding='utf-8') as f:
        json.dump(train_data, f, indent=4, ensure_ascii=False)
    print(f"💾 已保存 train_data 至: {train_output_path}")

    # 保存为 val_data.json
    val_output_path = os.path.join(output_dir, "val_data.json")
    with open(val_output_path, 'w', encoding='utf-8') as f:
        json.dump(val_data, f, indent=4, ensure_ascii=False)
    print(f"💾 已保存 val_data 至: {val_output_path}")

except Exception as e:
    print(f"❌ 文件保存失败: {e}")

In [ ]:
def load_and_index_data(json_file_path):
    """
    加载 JSON 数据，生成向量并建立 FAISS 索引
    
    Args:
        json_file_path: JSON 文件路径
    """
    
    print("开始加载数据...")
    # 1. 加载 JSON 数据
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"数据加载完成，共 {len(data)} 条记录")
    
    # 假设 JSON 是一个字典列表，且每条记录包含 'text' 字段
    texts = [item['text'] for item in data if 'text' in item]
    
    print(f"提取出 {len(texts)} 条文本用于向量化")
    
    # 2. 初始化嵌入模型 (all-MiniLM-L6-v2 - 384维)
    print("初始化嵌入模型 (all-MiniLM-L6-v2)...")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # 3. 批量生成向量 (使用 GPU 加速，如果可用)
    print("正在生成文本向量...")
    embeddings = model.encode(
        texts, 
        batch_size=32,  # 可根据内存调整
        show_progress_bar=True,
        convert_to_numpy=True
    )
    
    # 确保向量是 float32 类型（FAISS 标准）
    embeddings = embeddings.astype('float32')
    print(f"向量生成完成，形状: {embeddings.shape}")
    
    # 4. 构建 FAISS 索引 (使用 L2 距离 - 欧几里得距离)
    dimension = embeddings.shape[1]  # 384
    index = faiss.IndexFlatL2(dimension)  # 或者可以考虑 IndexFlatIP (内积) 用于余弦相似度
    
    # 将向量添加到索引中
    print("正在构建 FAISS 索引...")
    index.add(embeddings)
    
    print(f"FAISS 索引构建完成，包含 {index.ntotal} 个向量")
    
    # 5. 保存索引和原始数据映射
    base_dir = os.path.dirname(json_file_path)
    
    # 保存 FAISS 索引
    index_path = os.path.join(base_dir, "faiss_index.bin")
    faiss.write_index(index, index_path)
    print(f"FAISS 索引已保存至: {index_path}")
    
    # 保存原始数据，用于检索时获取原文本等信息
    original_data_path = os.path.join(base_dir, "original_data.pkl")
    with open(original_data_path, 'wb') as f:
        pickle.dump(data, f)
    print(f"原始数据已保存至: {original_data_path}")
    
    # 6. 保存文本列表（可选，方便检索时直接对应）
    text_list_path = os.path.join(base_dir, "text_list.pkl")
    with open(text_list_path, 'wb') as f:
        pickle.dump(texts, f)
    print(f"文本列表已保存至: {text_list_path}")
    
    print("\n--- 索引构建流程完成 ---")
    print(f"1. 向量维度: {dimension}")
    print(f"2. 索引类型: FAISS IndexFlatL2")
    print(f"3. 向量数量: {index.ntotal}")
    print(f"4. 索引文件: {index_path}")
    print(f"5. 原始数据文件: {original_data_path}")
    print(f"6. 文本列表文件: {text_list_path}")

def search_example(query_text, index_path, original_data_path, top_k=5):
    """
    示例：如何使用构建好的索引进行搜索
    
    Args:
        query_text: 查询文本
        index_path: FAISS 索引文件路径
        original_data_path: 原始数据文件路径
        top_k: 返回最相似的前 k 个结果
    """
    # 加载模型
    model = SentenceTransformer('all-MiniLM-L6-v2')
    # 编码查询
    query_embedding = model.encode([query_text], convert_to_numpy=True).astype('float32')
    
    # 加载索引
    index = faiss.read_index(index_path)
    # 加载原始数据
    with open(original_data_path, 'rb') as f:
        original_data = pickle.load(f)
    
    # 搜索
    distances, indices = index.search(query_embedding, k=top_k)
    
    print(f"\n查询: '{query_text}'")
    print(f"最相似的 {top_k} 条结果:")
    for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        print(f"{i+1}. 索引: {idx}, 距离: {dist:.4f}")
        print(f"   文本: {original_data[idx]['text'][:200]}...") # 显示前200字符
        if 'label' in original_data[idx]:
            print(f"   标签: {original_data[idx]['label']}")
        print("-" * 50)

if __name__ == "__main__":
    json_file_path = r"C:\Users\28995\Desktop\Heavenly Lake -2\split_output\train_data.json"
    
    # 执行索引构建
    load_and_index_data(json_file_path)
    
    # 可选：运行一个示例搜索来验证
    # query = "your example query here"
    # search_example(query, 
    #               r"C:\Users\28995\Desktop\Heavenly Lake -2\split_output\faiss_index.bin", 
    #               r"C:\Users\28995\Desktop\Heavenly Lake -2\split_output\original_data.pkl")

In [1]:
import json
import os
import time
from typing import List, Dict, Any, Optional, Tuple
from tqdm import tqdm
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import faiss
import pickle

# =============== 领域规范 ===============
NER_TYPES = {
    "CROP": ("作物", "Crop"),
    "VAR": ("品种", "Variety/Cultivar"),
    "TRT": ("性状", "Trait"),
    "GST": ("生育时期", "Growth Stage"),
    "GENE": ("基因", "Gene"),
    "QTL": ("数量性状位点", "QTL"),
    "MRK": ("分子标记", "Molecular Marker"),
    "CHR": ("染色体", "Chromosome"),
    "BM": ("育种方法", "Breeding Method"),
    "CROSS": ("亲本/杂交组合", "Parent/Cross"),
    "ABS": ("非生物胁迫", "Abiotic Stress"),
    "BIS": ("生物胁迫", "Biotic Stress")
}

RE_RELATIONS = {
    "CON": {"name": "包含", "desc": "品种属于某作物", "head": ["VAR"], "tail": ["CROP"]},
    "USE": {"name": "采用", "desc": "品种采用某种育种方法", "head": ["VAR"], "tail": ["BM"]},
    "HAS": {"name": "具有", "desc": "品种具备或关注某性状", "head": ["VAR"], "tail": ["TRT"]},
    "AFF": {"name": "影响", "desc": "非生物胁迫、基因、分子标记或 QTL 影响性状", "head": ["ABS", "GENE", "MRK", "QTL"], "tail": ["TRT"]},
    "OCI": {"name": "发生于", "desc": "性状或胁迫发生于某生育时期", "head": ["TRT", "ABS", "BIS"], "tail": ["GST"]},
    "LOI": {"name": "定位于", "desc": "分子标记、QTL 或基因定位于染色体或区间", "head": ["MRK", "QTL", "GENE"], "tail": ["CHR", "QTL"]}
}

VALID_ENTITY_LABELS = set(NER_TYPES.keys())
VALID_RELATION_LABELS = set(RE_RELATIONS.keys())


class RAGNERREPredictor:
    def __init__(self, api_key: str, faiss_index_path: str, original_data_path: str,
                 base_url: str = "https://api.xiaomimimo.com/v1"):
        """初始化预测器"""
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model = SentenceTransformer('all-MiniLM-L6-v2', cache_folder="./cache")
        self.index = faiss.read_index(faiss_index_path)
        with open(original_data_path, 'rb') as f:
            self.original_data = pickle.load(f)

    def _normalize_label(self, label: str, label_type: str) -> Optional[str]:
        """归一化标签（实体 or 关系）"""
        if not label:
            return None
        label = label.strip().upper()
        if label_type == "entity":
            if label in VALID_ENTITY_LABELS:
                return label
            for k, (cn, en) in NER_TYPES.items():
                if label in [cn, en, cn.upper(), en.upper(), k]:
                    return k
        elif label_type == "relation":
            if label in VALID_RELATION_LABELS:
                return label
            for k, info in RE_RELATIONS.items():
                if label in [info["name"], info["name"].upper(), k]:
                    return k
        return None

    def _find_text_position(self, text: str, input_text: str,
                            rough_start: int) -> Optional[Tuple[int, int]]:
        """
        在原文中基于文本内容查找真实位置。
        策略：
        1. 全局精确匹配 input_text.find(text)
        2. 若失败，在 rough_start ±20 范围内搜索
        3. 若仍失败，返回 None
        """
        if not text:
            return None
        # 策略1: 全局精确匹配
        pos = input_text.find(text)
        if pos != -1:
            return (pos, pos + len(text))
        # 策略2: 在 rough_start 附近 ±20 范围内搜索
        search_start = max(0, rough_start - 20)
        search_end = min(len(input_text), rough_start + 20 + len(text))
        window = input_text[search_start:search_end]
        pos_in_window = window.find(text)
        if pos_in_window != -1:
            real_pos = search_start + pos_in_window
            return (real_pos, real_pos + len(text))
        return None

    def _validate_entity(self, ent: Dict[str, Any],
                         input_text: str) -> Optional[Dict[str, Any]]:
        """验证单个实体（修复版：基于文本匹配修正位置）"""
        if not isinstance(ent, dict):
            return None
        start = ent.get("start")
        end = ent.get("end")
        text = ent.get("text")
        label = ent.get("label")
        if start is None or end is None or text is None or label is None:
            return None
        if not isinstance(text, str) or not text.strip():
            return None
        try:
            start, end = int(start), int(end)
        except (ValueError, TypeError):
            return None
        if not (0 <= start <= len(input_text)):
            return None

        # --- 修复点1: 基于文本匹配修正位置 ---
        if input_text[start:end] != text:
            corrected = self._find_text_position(text, input_text, start)
            if corrected is None:
                return None
            start, end = corrected

        if not (0 <= start < end <= len(input_text)):
            return None
        if input_text[start:end] != text:
            return None

        norm_label = self._normalize_label(label, "entity")
        if not norm_label:
            return None
        return {"start": start, "end": end, "text": text, "label": norm_label}

    def _validate_relation(self, rel: Dict[str, Any],
                           entities: List[Dict[str, Any]],
                           input_text: str) -> Optional[Dict[str, Any]]:
        """验证单个关系（修复版：基于文本匹配修正 head/tail 位置）"""
        if not isinstance(rel, dict):
            return None
        required = ["head", "tail", "head_type", "tail_type", "label"]
        if not all(k in rel for k in required):
            return None
        head_text_raw = rel.get("head", "")
        tail_text_raw = rel.get("tail", "")
        if not head_text_raw or not tail_text_raw:
            return None

        # --- 修复点2: 基于文本匹配修正 head/tail 位置 ---
        h_pos = input_text.find(head_text_raw)
        t_pos = input_text.find(tail_text_raw)
        if h_pos == -1 or t_pos == -1:
            return None
        h_s, h_e = h_pos, h_pos + len(head_text_raw)
        t_s, t_e = t_pos, t_pos + len(tail_text_raw)
        if not (0 <= h_s < h_e <= len(input_text) and 0 <= t_s < t_e <= len(input_text)):
            return None

        head_type = self._normalize_label(rel["head_type"], "entity")
        tail_type = self._normalize_label(rel["tail_type"], "entity")
        rel_label = self._normalize_label(rel["label"], "relation")
        if not head_type or not tail_type or not rel_label:
            return None

        rel_def = RE_RELATIONS[rel_label]
        if isinstance(rel_def["head"], list) and head_type not in rel_def["head"]:
            return None
        if isinstance(rel_def["tail"], list) and tail_type not in rel_def["tail"]:
            return None

        return {
            "head": head_text_raw,
            "head_start": h_s,
            "head_end": h_e,
            "head_type": head_type,
            "tail": tail_text_raw,
            "tail_start": t_s,
            "tail_end": t_e,
            "tail_type": tail_type,
            "label": rel_label
        }

    def retrieve_similar_examples(self, query_text: str,
                                  top_k: int = 3) -> List[Dict[str, Any]]:
        """检索相似示例"""
        if not query_text.strip():
            return []
        emb = self.model.encode([query_text], convert_to_numpy=True).astype('float32')
        distances, indices = self.index.search(emb, k=top_k)
        retrieved = []
        for idx in indices[0]:
            if 0 <= idx < len(self.original_data):
                retrieved.append(self.original_data[idx])
        return retrieved

    def build_prompt(self, input_text: str,
                     examples: List[Dict[str, Any]]) -> str:
        """构建提示词"""
        prompt_lines = [
            "你是一名农业遗传育种领域的专业信息抽取专家。",
            "请严格根据以下规范，从输入文本中识别命名实体（NER）和实体间关系（RE）：",
            "",
            "=== 实体类型（12类） ==="
        ]
        for lbl, (cn, en) in NER_TYPES.items():
            prompt_lines.append(f"- {lbl}: {cn} ({en})")
        prompt_lines.extend(["", "=== 关系类型（6类） ==="])
        for rid, info in RE_RELATIONS.items():
            prompt_lines.append(f"- {rid} ({info['name']}): {info['desc']}")
        prompt_lines.extend([
            "",
            "=== 输出格式要求 ===",
            "- 仅输出一个 JSON 对象，包含 text、entities 和 relations 三个键，无任何额外说明。",
            "- text: 输入的原文本",
            "- entities: [{\"start\":int, \"end\":int, \"text\":str, \"label\":str}]",
            "- relations: [{\"head\":str, \"head_start\":int, \"head_end\":int, \"head_type\":str, \"tail\":str, \"tail_start\":int, \"tail_end\":int, \"tail_type\":str, \"label\":str}]",
            "- 所有位置为字符级偏移（0-indexed），且必须在文本长度内。",
            "- 实体 text 必须在原文中完全匹配对应位置，start 和 end 必须是精确的字符偏移量，请仔细核对。",
            "- 若无法识别，返回空列表 []。",
            ""
        ])
        for i, ex in enumerate(examples[:3], 1):
            prompt_lines.extend([
                f"\n示例 {i}：",
                f"文本：\"{ex['text']}\"",
                "输出：```json",
                json.dumps({
                    "text": ex['text'],
                    "entities": ex.get("entities", []),
                    "relations": ex.get("relations", [])
                }, ensure_ascii=False, indent=2),
                "```"
            ])
        prompt_lines.extend([
            "",
            "=== 待处理文本 ===",
            f"文本：\"{input_text}\"",
            "",
            "请严格按上述格式输出 JSON："
        ])
        return "\n".join(prompt_lines)

    def call_llm(self, prompt: str, input_text: str,
                 max_retries: int = 3) -> Dict[str, Any]:
        """调用 LLM，带重试机制 & 安全解析"""
        for attempt in range(1, max_retries + 1):
            try:
                response = self.client.chat.completions.create(
                    model='mimo-v2.5-pro',
                    messages=[{"role": "user", "content": prompt}],
                    response_format={"type": "json_object"},
                    max_tokens=4096,
                    temperature=0.05,
                    top_p=0.85,
                    seed=42
                )
                content = response.choices[0].message.content
                for delim in ["```json", "```"]:
                    if delim in content:
                        start = content.find(delim) + len(delim)
                        end = content.find("```", start)
                        if end == -1:
                            end = len(content)
                        content = content[start:end].strip()
                        break
                try:
                    raw_result = json.loads(content)
                except json.JSONDecodeError:
                    content = content.strip().rstrip(',').rstrip('}')
                    content = content.replace('\u201c', '"').replace('\u201d', '"')
                    content = content.replace("\u2018", "'").replace("\u2019", "'")
                    try:
                        raw_result = json.loads(content)
                    except Exception:
                        raise ValueError("JSON 解析失败：内容格式严重异常")

                result = {"text": input_text, "entities": [], "relations": []}

                # --- 实体去重 ---
                seen_entities = set()
                for ent in raw_result.get("entities", []):
                    validated = self._validate_entity(ent, input_text)
                    if validated:
                        key = (validated["start"], validated["end"], validated["label"])
                        if key not in seen_entities:
                            seen_entities.add(key)
                            result["entities"].append(validated)

                for rel in raw_result.get("relations", []):
                    validated = self._validate_relation(rel, result["entities"], input_text)
                    if validated:
                        result["relations"].append(validated)

                return result
            except Exception as e:
                print(f"[EXCEPTION] 第 {attempt} 次调用异常: {type(e).__name__}: {e}")
                if attempt < max_retries:
                    time.sleep(2 ** attempt)
                continue

        print("[FATAL] LLM 调用全部失败，返回空结果")
        return {"text": input_text, "entities": [], "relations": []}

    def post_process_align(self, result: Dict[str, Any]) -> Dict[str, Any]:
        """
        后处理：对所有预测结果做全局文本对齐校验，剔除仍然错位的实体/关系。
        """
        input_text = result.get("text", "")
        if not input_text:
            return result

        # --- 实体全局对齐 ---
        aligned_entities = []
        seen_keys = set()
        for ent in result.get("entities", []):
            start = ent.get("start")
            end = ent.get("end")
            text = ent.get("text", "")
            label = ent.get("label", "")
            if start is None or end is None or not text:
                continue
            try:
                start, end = int(start), int(end)
            except (ValueError, TypeError):
                continue
            if start < 0 or end > len(input_text) or start > end:
                corrected = self._find_text_position(text, input_text, start)
                if corrected is None:
                    continue
                start, end = corrected
            if input_text[start:end] != text:
                corrected = self._find_text_position(text, input_text, start)
                if corrected is None:
                    continue
                start, end = corrected
            if not (0 <= start < end <= len(input_text)):
                continue
            if input_text[start:end] != text:
                continue
            key = (start, end, label)
            if key in seen_keys:
                continue
            seen_keys.add(key)
            aligned_entities.append({
                "start": start, "end": end, "text": text, "label": label
            })

        # --- 关系全局对齐 ---
        aligned_relations = []
        for rel in result.get("relations", []):
            head_text = rel.get("head", "")
            tail_text = rel.get("tail", "")
            if not head_text or not tail_text:
                continue
            h_pos = input_text.find(head_text)
            t_pos = input_text.find(tail_text)
            if h_pos == -1 or t_pos == -1:
                continue
            h_s, h_e = h_pos, h_pos + len(head_text)
            t_s, t_e = t_pos, t_pos + len(tail_text)
            if not (0 <= h_s < h_e <= len(input_text) and 0 <= t_s < t_e <= len(input_text)):
                continue
            aligned_relations.append({
                "head": head_text,
                "head_start": h_s,
                "head_end": h_e,
                "head_type": rel.get("head_type", ""),
                "tail": tail_text,
                "tail_start": t_s,
                "tail_end": t_e,
                "tail_type": rel.get("tail_type", ""),
                "label": rel.get("label", "")
            })

        return {
            "text": input_text,
            "entities": aligned_entities,
            "relations": aligned_relations
        }

    def predict_single(self, input_text: str, top_k: int = 3) -> Dict[str, Any]:
        """预测单条文本"""
        if not input_text or not input_text.strip():
            return {"text": input_text, "entities": [], "relations": []}
        examples = self.retrieve_similar_examples(input_text, top_k=top_k)
        prompt = self.build_prompt(input_text, examples)
        return self.call_llm(prompt, input_text=input_text)


def predict_test_a_file(
    test_file_path: str,
    api_key: str,
    output_path: str = None,
    batch_size: int = 1,
    base_url: str = "https://api.xiaomimimo.com/v1"
) -> List[Dict[str, Any]]:
    """批量预测 test_A.json 文件"""
    if not os.path.exists(test_file_path):
        raise FileNotFoundError(f"测试文件不存在: {test_file_path}")

    base_dir = os.path.dirname(test_file_path)
    faiss_index_path = os.path.join(base_dir, "faiss_index.bin")
    original_data_path = os.path.join(base_dir, "original_data.pkl")

    for path in [faiss_index_path, original_data_path]:
        if not os.path.exists(path):
            raise FileNotFoundError(f"依赖文件缺失: {path}")

    predictor = RAGNERREPredictor(api_key, faiss_index_path, original_data_path, base_url)

    with open(test_file_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)

    results = []
    total = len(test_data)

    print(f"开始预测 {total} 条记录...")
    for i, item in enumerate(tqdm(test_data, desc="进度")):
        try:
            text = None
            if isinstance(item, dict):
                text = item.get("text") or item.get("Text") or item.get("sentence") or item.get("content")
                if not text:
                    keys = [k for k in item.keys() if k.lower() in ['text', 'content', 'abstract', 'paragraph']]
                    if keys:
                        text = item[keys[0]]
            if not text:
                text = str(item)

            if not text.strip():
                result = {"text": "", "entities": [], "relations": [], "error": "空输入"}
            else:
                pred_res = predictor.predict_single(text, top_k=3)
                # --- 修复点5: 后处理全局对齐 ---
                result = predictor.post_process_align(pred_res)
            results.append(result)

        except Exception as e:
            error_msg = f"第 {i+1} 条处理异常: {type(e).__name__}: {e}"
            print(f"[ERROR] {error_msg}")
            # --- 修复点3: 对 dict 使用 .get() 而非 getattr ---
            results.append({
                "text": (item.get('text', str(item)) if isinstance(item, dict) else str(item))[:100],
                "entities": [],
                "relations": [],
                "error": str(e)
            })

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"\n✅ 预测完成！结果已保存至: {output_path}")

    return results


# ====================== 主程序入口 ======================
if __name__ == "__main__":
    YOUR_API_KEY = os.getenv("MIMO_API_KEY", "sk-cv181nd9t5ofudohhl2hlt28gy3jkbqwcp1vjg38o36sbi0e")
    base_url = "https://api.xiaomimimo.com/v1"

    test_file_path = r"C:\Users\28995\Desktop\Heavenly Lake -2\test_A.json"
    output_path = r"C:\Users\28995\Desktop\Heavenly Lake -2\submit.json"

    try:
        results = predict_test_a_file(
            test_file_path=test_file_path,
            api_key=YOUR_API_KEY,
            output_path=output_path,
            batch_size=1,
            base_url=base_url
        )
        print(f"\n📊 总计处理 {len(results)} 条，成功 {sum(1 for r in results if 'error' not in r)} 条")

        for i in range(min(2, len(results))):
            r = results[i]
            print(f"\n--- 示例 {i+1} ---")
            print("原文:", r["text"][:80] + "..." if len(r["text"]) > 80 else r["text"])
            print("实体数:", len(r["entities"]))
            print("关系数:", len(r["relations"]))
            if r.get("error"):
                print("⚠️ 错误:", r["error"])

    except Exception as e:
        print(f"❌ 主程序崩溃: {e}")
        import traceback
        traceback.print_exc()


e:\anacond\envs\py310\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8386.33it/s]


开始预测 400 条记录...


进度:  13%|█▎        | 51/400 [14:28<1:45:30, 18.14s/it]

[EXCEPTION] 第 1 次调用异常: ValueError: JSON 解析失败：内容格式严重异常
[EXCEPTION] 第 2 次调用异常: ValueError: JSON 解析失败：内容格式严重异常


进度: 100%|██████████| 400/400 [1:40:11<00:00, 15.03s/it]


✅ 预测完成！结果已保存至: C:\Users\28995\Desktop\Heavenly Lake -2\submit.json

📊 总计处理 400 条，成功 400 条

--- 示例 1 ---
原文: The study investigated the molecular mechanism of two Tartary buckwheat genotype...
实体数: 6
关系数: 1

--- 示例 2 ---
原文: Autotetraploid sorghum inbreds have higher kernel weight, seed yield, and protei...
实体数: 9
关系数: 6
